# TrueID Live Commerce Copilot - Captioning Demo

This notebook runs the captioning and transcript-driven commerce-action slices for a live-commerce stream. It clones the GitHub repo when run from a blank Colab runtime. The default path tries the configured ASR model and falls back to a cached audio-1 transcript so the demo completes without API keys, GPU, ffmpeg, or speech-model downloads. To try Typhoon Whisper, set `ASR_PROVIDER = "typhoon_whisper"` and choose `ASR_MODEL` from `"turbo"`, `"large-v3"`, `"medium"`, or `"isan-medium"`. Typhoon ASR is run on timed audio windows so captions can stream in real time.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Siratish/Live-Commerce-Copilot.git"
REPO_DIR_NAME = "Live-Commerce-Copilot"

def looks_like_project(path: Path) -> bool:
    return (path / "config" / "demo.yaml").exists() and (path / "src").exists()

repo_root = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if looks_like_project(candidate):
        repo_root = candidate
        print(f"Using existing repo checkout: {repo_root}")
        break

if repo_root is None:
    clone_parent = Path("/content") if Path("/content").exists() else Path.cwd()
    repo_root = clone_parent / REPO_DIR_NAME
    if repo_root.exists() and not looks_like_project(repo_root):
        raise RuntimeError(f"{repo_root} exists but does not look like the target project.")
    if not repo_root.exists():
        subprocess.check_call(["git", "clone", REPO_URL, str(repo_root)])
    else:
        print(f"Using existing clone: {repo_root}")

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print(f"Active repo root: {repo_root}")

requirements = repo_root / "requirements.txt"
if requirements.exists():
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)])
    except subprocess.CalledProcessError as exc:
        print(f"Dependency install failed, continuing with stdlib fallback: {exc}")
else:
    print("requirements.txt not found; continuing with the current runtime")

In [ ]:
from pathlib import Path
import json

from src.pipeline.run_captioning_demo import run_from_config

ASR_PROVIDER = None  # Example: "typhoon_whisper"
ASR_MODEL = None  # Example: "turbo", "large-v3", "medium", or "isan-medium"
ASR_CHUNK_LENGTH_SECONDS = None  # Example: 4 for real-time Typhoon caption windows
ASR_MAX_NEW_TOKENS = None  # Typhoon Whisper Turbo default is safely clamped to 440

summary = run_from_config(
    Path("config/demo.yaml"),
    asr_provider_override=ASR_PROVIDER,
    asr_model_override=ASR_MODEL,
    asr_chunk_length_seconds_override=ASR_CHUNK_LENGTH_SECONDS,
    asr_max_new_tokens_override=ASR_MAX_NEW_TOKENS,
)
summary

In [ ]:
caption_path = Path(summary["outputs"]["json"])
captions = json.loads(caption_path.read_text(encoding="utf-8"))

rows = []
for segment in captions["segments"]:
    rows.append({
        "start": segment["start"],
        "end": segment["end"],
        "seconds": round(segment["end"] - segment["start"], 2),
        "text": segment["text"],
        "source": segment["source"],
    })

try:
    import pandas as pd
    display(pd.DataFrame(rows))
except Exception:
    rows

## Live Caption Playback

Press play below. The generated caption text updates as the audio timeline reaches each segment.

In [ ]:
from IPython.display import HTML, display
from src.utils.realtime_caption import build_realtime_caption_html

audio_path = Path(summary["audio_path"]) if summary.get("audio_path") else Path("data/demo/audio/1.mp3")
display(HTML(build_realtime_caption_html(audio_path, captions)))

## Commerce Action Timeline

The next cell turns transcript events into product cards, price drops, promo badges, bundle recommendations, and flash-sale countdowns.

## Optional Typhoon2.5 Decision Smoke Test

Set `DECISION_PROVIDER = "typhoon25"` below and run this cell before the action timeline. It checks whether the 4B model can answer one short prompt. If the smoke test does not pass, the action cell will use the deterministic decider.

In [ ]:
DECISION_PROVIDER = "deterministic"  # Example: "typhoon25"; "typhoon_s" still works as a legacy alias
DECISION_MODEL = "scb10x/typhoon2.5-qwen3-4b"
DECISION_MAX_NEW_TOKENS = 256
DECISION_SMOKE_MAX_NEW_TOKENS = 32
DECISION_TEMPERATURE = 0.1
DECISION_SMOKE_TIMEOUT_SECONDS = 180
INSTALL_DECISION_AI_DEPS = False
KEEP_TYPHOON25_MODEL_LOADED_FOR_ACTIONS = False
RELEASE_TYPHOON25_AFTER_ACTIONS = True

TYPHOON25_MODEL_READY = DECISION_PROVIDER not in {"typhoon25", "typhoon_s"}
TYPHOON25_SMOKE_PASSED = DECISION_PROVIDER not in {"typhoon25", "typhoon_s"}
TYPHOON25_SMOKE_RESPONSE = None
typhoon25_tokenizer = None
typhoon25_model = None
typhoon25_torch = None

def release_typhoon25_resources():
    global typhoon25_model, typhoon25_tokenizer, typhoon25_torch, TYPHOON25_MODEL_READY
    import gc
    typhoon25_model = None
    typhoon25_tokenizer = None
    TYPHOON25_MODEL_READY = False
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
            print(f"CUDA allocated after cleanup: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    except Exception as exc:
        print(f"CUDA cleanup skipped: {exc}")

def _generation_kwargs(max_new_tokens):
    kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": DECISION_TEMPERATURE > 0,
        "repetition_penalty": 1.05,
    }
    if DECISION_TEMPERATURE > 0:
        kwargs["temperature"] = DECISION_TEMPERATURE
    return kwargs

def typhoon25_generate_text(messages, max_new_tokens=None):
    inputs = typhoon25_tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(typhoon25_model.device)
    outputs = typhoon25_model.generate(
        **inputs,
        **_generation_kwargs(max_new_tokens or DECISION_MAX_NEW_TOKENS),
    )
    response = outputs[0][inputs["input_ids"].shape[-1]:]
    return typhoon25_tokenizer.decode(response, skip_special_tokens=True)

if DECISION_PROVIDER in {"typhoon25", "typhoon_s"}:
    import sys
    import subprocess
    import time
    start_time = time.perf_counter()
    try:
        if INSTALL_DECISION_AI_DEPS:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-ai.txt"])

        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer
        typhoon25_torch = torch

        if not torch.cuda.is_available():
            raise RuntimeError("CUDA GPU is not available. Typhoon2.5 4B may be too slow for this notebook on CPU.")

        try:
            import signal
        except Exception:
            signal = None

        class _SmokeTimeout(Exception):
            pass

        def _alarm_handler(signum, frame):
            raise _SmokeTimeout(f"Typhoon2.5 smoke test exceeded {DECISION_SMOKE_TIMEOUT_SECONDS} seconds")

        use_alarm = bool(signal and hasattr(signal, "SIGALRM"))
        if use_alarm:
            signal.signal(signal.SIGALRM, _alarm_handler)
            signal.alarm(DECISION_SMOKE_TIMEOUT_SECONDS)

        try:
            typhoon25_tokenizer = AutoTokenizer.from_pretrained(DECISION_MODEL)
            typhoon25_model = AutoModelForCausalLM.from_pretrained(
                DECISION_MODEL,
                torch_dtype=torch.bfloat16,
                device_map="auto",
            )
            smoke_messages = [
                {"role": "system", "content": "Return only compact JSON."},
                {"role": "user", "content": "ตอบกลับเป็น JSON เท่านั้น: {\"ok\": true, \"message\": \"ready\"}"},
            ]
            TYPHOON25_SMOKE_RESPONSE = typhoon25_generate_text(smoke_messages, DECISION_SMOKE_MAX_NEW_TOKENS)
            if not str(TYPHOON25_SMOKE_RESPONSE).strip():
                raise RuntimeError("Typhoon2.5 returned an empty smoke-test response")
        finally:
            if use_alarm:
                signal.alarm(0)

        TYPHOON25_SMOKE_PASSED = True
        TYPHOON25_MODEL_READY = KEEP_TYPHOON25_MODEL_LOADED_FOR_ACTIONS
        print(f"Typhoon2.5 smoke test passed in {time.perf_counter() - start_time:.1f}s")
        print(TYPHOON25_SMOKE_RESPONSE)
        if KEEP_TYPHOON25_MODEL_LOADED_FOR_ACTIONS:
            print("Typhoon2.5 model remains loaded for the action timeline cell.")
        else:
            release_typhoon25_resources()
            print("Typhoon2.5 model was unloaded after the smoke test.")
    except Exception as exc:
        TYPHOON25_MODEL_READY = False
        TYPHOON25_SMOKE_PASSED = False
        release_typhoon25_resources()
        print(f"Typhoon2.5 smoke test failed: {exc}")
        print("The action timeline cell will use the deterministic decision provider.")
else:
    print("Skipping Typhoon2.5 smoke test because DECISION_PROVIDER is deterministic.")

TYPHOON25_MODEL_READY

In [ ]:
from src.ai.decision import Typhoon25DecisionProvider
from src.pipeline.run_commerce_actions import run_actions_from_paths

effective_decision_provider = DECISION_PROVIDER
decision_provider_instance = None
if DECISION_PROVIDER in {"typhoon25", "typhoon_s"}:
    if TYPHOON25_MODEL_READY and typhoon25_model is not None and typhoon25_tokenizer is not None:
        decision_provider_instance = Typhoon25DecisionProvider(
            model_id=DECISION_MODEL,
            max_new_tokens=DECISION_MAX_NEW_TOKENS,
            temperature=DECISION_TEMPERATURE,
            text_generator=typhoon25_generate_text,
        )
    else:
        effective_decision_provider = "deterministic"
        print("Typhoon2.5 was not verified by the smoke test; using deterministic decisions.")

try:
    actions_summary = run_actions_from_paths(
        captions_path=caption_path,
        catalog_path=Path("data/demo/product_catalog.csv"),
        promotions_path=Path("data/demo/promotions.csv"),
        output_dir=Path("outputs"),
        audio_path=audio_path,
        decision_provider=decision_provider_instance,
        decision_provider_name=effective_decision_provider,
        decision_model=DECISION_MODEL if effective_decision_provider in {"typhoon25", "typhoon_s"} else None,
        decision_max_new_tokens=DECISION_MAX_NEW_TOKENS,
        decision_temperature=DECISION_TEMPERATURE,
    )
finally:
    if DECISION_PROVIDER in {"typhoon25", "typhoon_s"} and RELEASE_TYPHOON25_AFTER_ACTIONS:
        if decision_provider_instance is not None:
            decision_provider_instance = None
        release_typhoon25_resources()
        print("Typhoon2.5 model resources were released after action extraction.")
actions_summary["decision_provider"], actions_summary["action_count"], actions_summary["outputs"]

In [ ]:
action_rows = []
for action in actions_summary["actions"]:
    action_rows.append({
        "time": action["timestamp"],
        "action": action["action_type"],
        "skus": " + ".join(action["skus"]),
        "confidence": action["confidence"],
    })

try:
    import pandas as pd
    display(pd.DataFrame(action_rows))
except Exception:
    action_rows

In [ ]:
display(HTML(Path(actions_summary["outputs"]["html"]).read_text(encoding="utf-8")))

In [ ]:
duration = captions.get("duration_seconds") or captions["segments"][-1]["end"]
bars = []
for index, segment in enumerate(captions["segments"], start=1):
    left = 100 * segment["start"] / duration
    width = 100 * (segment["end"] - segment["start"]) / duration
    bars.append(
        f'<div style="position:relative;height:26px;margin:4px 0;background:#f2f4f7;border-radius:4px;">'
        f'<div title="Segment {index}: {segment["text"]}" style="position:absolute;left:{left:.2f}%;width:{width:.2f}%;height:100%;background:#e51b23;border-radius:4px;"></div>'
        f'<span style="position:absolute;left:8px;top:4px;font:12px Arial;color:#111;">{index}</span>'
        f'</div>'
    )
html = '<h3>Caption Coverage Timeline</h3>' + ''.join(bars)
try:
    from IPython.display import HTML, display
    display(HTML(html))
except Exception:
    print(html)

In [ ]:
metrics_path = Path(summary["outputs"]["metrics"])
metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
metrics

## Production Note

This first slice uses cached/file input for reliability. In production, the same `CaptioningEngine` interface can receive overlapping four-second audio chunks from an RTMP, HLS, or WebRTC adapter, publish WebVTT segments to the viewer overlay, and persist transcript events for moderator and post-show analytics.